# 03 Modeling Baseline and Graph-Enhanced Models

Goal:
- Compare raw transaction features against graph-enhanced features.
- Train Logistic Regression, Random Forest, XGBoost, and LightGBM when available.

In [1]:
from pathlib import Path
import sys
sys.path.append("..")

import joblib
import pandas as pd

from src.data.load_data import chronological_split
from src.features.tabular_features import make_model_matrix
from src.models.train_models import train_and_evaluate

In [2]:
DATA_PATH = Path("../data/processed/hi_small_features.parquet")
df = pd.read_parquet(DATA_PATH)
train_df, val_df, test_df = chronological_split(df, train_size=0.60, val_size=0.20)

print(train_df.shape, val_df.shape, test_df.shape)
print("Train label rate:", train_df["is_laundering"].mean())
print("Test label rate:", test_df["is_laundering"].mean())

(300000, 33) (100000, 33) (100000, 33)
Train label rate: 5e-05
Test label rate: 0.00149


In [3]:
# Experiment 1: raw transaction features
X_train_raw, y_train = make_model_matrix(train_df, include_graph_features=False)
X_test_raw, y_test = make_model_matrix(test_df, include_graph_features=False)

# Align columns
X_test_raw = X_test_raw.reindex(columns=X_train_raw.columns, fill_value=0)

raw_metrics, raw_models = train_and_evaluate(X_train_raw, y_train, X_test_raw, y_test)
raw_metrics["feature_set"] = "raw"
raw_metrics

Training logistic_regression...


/home/acer/dev/aml-graph-dm/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training random_forest...
Training xgboost...
Training lightgbm...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 15, number of negative: 299985
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.246383 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1353
[LightGBM] [Info] Number of data points in the train set: 300000, number of used features: 44
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

KeyboardInterrupt: 

In [4]:
# Experiment 2: raw + graph aggregate features
X_train_graph, y_train = make_model_matrix(train_df, include_graph_features=True)
X_test_graph, y_test = make_model_matrix(test_df, include_graph_features=True)

# Align columns
X_test_graph = X_test_graph.reindex(columns=X_train_graph.columns, fill_value=0)

graph_metrics, graph_models = train_and_evaluate(X_train_graph, y_train, X_test_graph, y_test)
graph_metrics["feature_set"] = "raw_plus_graph"
graph_metrics

Training logistic_regression...


/home/acer/dev/aml-graph-dm/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training random_forest...
Training xgboost...
Training lightgbm...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 15, number of negative: 299985
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.194907 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2553
[LightGBM] [Info] Number of data points in the train set: 300000, number of used features: 56
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


KeyboardInterrupt: 

In [6]:
all_metrics = pd.concat([raw_metrics, graph_metrics], ignore_index=True)
all_metrics = all_metrics.sort_values(["pr_auc", "f1_suspicious"], ascending=False)
display(all_metrics)

Path("../reports/tables").mkdir(parents=True, exist_ok=True)
all_metrics.to_csv("../reports/tables/model_metrics.csv", index=False)

# Save the best graph-enhanced model if it exists; otherwise save best raw model.
best_row = all_metrics.iloc[0]
best_set = best_row["feature_set"]
best_model_name = best_row["model"]
best_models = graph_models if best_set == "raw_plus_graph" else raw_models
best_model = best_models[best_model_name]

Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, "../models/best_model.joblib")
print("Saved best model:", best_model_name, best_set)

NameError: name 'raw_metrics' is not defined